In [ ]:
# Configure the Kaggle-mounted source repo and the Roboflow dataset to download
SRC_DIR = '/kaggle/input/fridge-detector'
DATA_DIR = '/kaggle/working/data/roboflow'

ROBOFLOW_WORKSPACE = 'practicum-ziryz'
ROBOFLOW_PROJECT = 'fridge-dataset-oi7ld'
ROBOFLOW_VERSION = 1
ROBOFLOW_SECRET_NAME = 'ROBOFLOW_API_KEY'

import os
import sys
from pathlib import Path

sys.path.insert(0, f'{SRC_DIR}/src')
os.environ['KAGGLE_SRC'] = SRC_DIR
os.environ['KAGGLE_DATA'] = DATA_DIR

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['ROBOFLOW_API_KEY'] = secrets.get_secret(ROBOFLOW_SECRET_NAME)
    print(f'Loaded Kaggle secret: {ROBOFLOW_SECRET_NAME}')
except Exception as exc:
    raise RuntimeError(
        'Could not load ROBOFLOW_API_KEY from Kaggle secrets. Add it under Add-ons > Secrets and enable Internet.'
    ) from exc

print('Python:', sys.version)
print('SRC_DIR exists:', os.path.isdir(SRC_DIR))
print('DATA_DIR target:', DATA_DIR)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
SRC_DIR exists:     True
IMAGES_DIR exists:  True
ANNOTS_DIR exists:  True


In [ ]:
# Install project dependencies for Kaggle runtime
!pip install -q rich pyyaml roboflow

# Kaggle preinstalls gcsfs pinned to this fsspec version; restore it to avoid runtime conflicts
!pip install -q "fsspec==2025.3.0"

# Optional: silence google-adk warning if your Kaggle image includes google-adk
!pip install -q "google-cloud-bigquery-storage>=2.0.0"

In [ ]:
# Download the Roboflow export into Kaggle working storage and validate the layout
import subprocess
from pathlib import Path

result = subprocess.run([
    'python', f'{SRC_DIR}/scripts/download_roboflow.py',
    '--workspace', ROBOFLOW_WORKSPACE,
    '--project', ROBOFLOW_PROJECT,
    '--version', str(ROBOFLOW_VERSION),
    '--output-dir', DATA_DIR,
], check=True)

data_yaml = Path(DATA_DIR) / 'data.yaml'
if not data_yaml.exists():
    raise RuntimeError(f'data.yaml not found after download: {data_yaml}')

print('Downloaded dataset root:', DATA_DIR)
print('data.yaml exists:', data_yaml.exists())
print('train exists:', (Path(DATA_DIR) / 'train').exists())
print('valid exists:', (Path(DATA_DIR) / 'valid').exists())

CUDA available: True
Device: Tesla T4
Dataset: 0 samples, 25 classes


In [ ]:
# Verify GPU and dataset layout, then start training
import os
import subprocess
import torch

print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

from data.dataset import RoboflowDetectionDataset, load_roboflow_data_config

class_names, split_dirs = load_roboflow_data_config(DATA_DIR)
train_dir = split_dirs['train']
val_dir = split_dirs.get('valid') or split_dirs.get('val')

train_ds = RoboflowDetectionDataset(
    str(train_dir),
    str(train_dir.with_name('labels')),
    class_names,
    image_size=512,
    augment=False,
 )

print(f'Classes: {len(class_names)}')
print('First 10 classes:', class_names[:10])
print('Train images:', train_dir)
print('Validation images:', val_dir)
print(f'Train samples: {len(train_ds)}')

env = os.environ.copy()
env['PYTHONPATH'] = f'{SRC_DIR}/src'

result = subprocess.run([
    'python', f'{SRC_DIR}/scripts/train.py',
    '--config', f'{SRC_DIR}/configs/kaggle.yaml',
    '--data-dir', DATA_DIR,
], env=env)

exit(result.returncode)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


╭───────────────────────── Fridge Detector — Training ─────────────────────────╮
│ Device:     cuda                                                             │
│ Backbone:   resnet50  fpn_channels=256                                       │
│ Dataset:    4453 train  494 val  25 classes                                  │
│ Params:     41.20M trainable                                                 │
╰──────────────────────────────────────────────────────────────────────────────╯
─────────────────────────── Epoch 1/50  lr=1.00e-04 ────────────────────────────

  rpn_obj_loss=0.045  rpn_box_loss=0.015  cls_loss=0.744  box_loss=0.016  
total=0.8196
  val:  62.6 dets/img  confidence=0.153
  ★ New best saved → /kaggle/working/checkpoints/best.pt
─────────────────────────── Epoch 2/50  lr=9.99e-05 ────────────────────────────

  rpn_obj_loss=0.022  rpn_box_loss=0.012  cls_loss=0.487  box_loss=0.018  
total=0.5395
  val:  48.8 dets/img  confidence=0.230
  ★ New best saved → /kaggle/working/

In [ ]:
# List saved checkpoints
import glob
import os

for ckpt in sorted(glob.glob('/kaggle/working/checkpoints/*.pt')):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'{ckpt}  ({size_mb:.1f} MB)')

/kaggle/working/checkpoints/best.pt  (166.2 MB)
/kaggle/working/checkpoints/latest.pt  (495.8 MB)
